# BOLT reproduction on BANKING77
Select an NVIDIA GPU runtime in Colab, then run the setup, GPU checks and reduced trial in order.
The notebook contains its input scripts. GPU results are kept in a separate directory from earlier CPU runs.

In [7]:
from pathlib import Path
import os
import subprocess
import sys
from google.colab import files

root = Path("/content/bolt-reproduction-gpu")
root.mkdir(exist_ok=True)
results = root / "results"
results.mkdir(exist_ok=True)
subprocess.run(["nvidia-smi"], check=True)
sources = {'analyze.py': '"""Build submission tables and pick reviewable errors from completed trials."""\n\nimport argparse\nfrom collections import Counter\nfrom pathlib import Path\n\nfrom reproduce import ROOT, read_rows, write_rows\n\n\ndef compare(output):\n    """Join BOLT\'s published scores and the completed author-code runs."""\n    references = {(row["method"], row["known_ratio"]): row\n                  for row in read_rows(ROOT / "published.csv")}\n    rows = []\n    for result in read_rows(output / "table_full.csv"):\n        reference = references[result["method"], result["known_ratio"]]\n        row = dict(method=result["method"], known_ratio=result["known_ratio"],\n                   detector=result["detector"], folds=result["folds"])\n        for metric in ("K-F1", "N-F1"):\n            row[f"published_{metric}"] = float(reference[metric])\n            row[f"reproduced_{metric}"] = float(result[metric])\n            row[f"delta_{metric}"] = round(float(result[metric]) - float(reference[metric]), 3)\n            row[f"fold_sd_{metric}"] = result[f"{metric}_fold_sd"]\n        rows.append(row)\n    write_rows(output / "table_comparison.csv", rows)\n\n\ndef mask_effect(output):\n    """Compare released and corrected KNNCon at matching settings."""\n    released = {(row["known_ratio"], row["detector"]): row\n                for row in read_rows(output / "table_full.csv") if row["method"] == "knncon"}\n    rows = []\n    for corrected in read_rows(output / "table_mask.csv"):\n        key = corrected["known_ratio"], corrected["detector"]\n        row = dict(known_ratio=key[0], detector=key[1])\n        for metric in ("K-F1", "N-F1"):\n            row[f"released_{metric}"] = float(released[key][metric])\n            row[f"corrected_{metric}"] = float(corrected[metric])\n            row[f"delta_{metric}"] = round(row[f"corrected_{metric}"] - row[f"released_{metric}"], 3)\n        rows.append(row)\n    write_rows(output / "table_mask_effect.csv", rows)\n\n\ndef errors(source, output):\n    """Count error categories and extract a deterministic example from each."""\n    tests = read_rows(source / "data/banking/origin_data/test.tsv", "\\t")\n    counts, examples = [], []\n    for mode in ("full", "mask"):\n        for trial in sorted((output / mode).glob("*/run.json")):\n            name, ratio, fold = trial.parent.name.split("_")\n            known_path = source / f"data/banking/label/fold5/part{fold}/label_known_{ratio}.list"\n            known = set(known_path.read_text(encoding="utf-8").splitlines())\n            for path in sorted(trial.parent.glob("predictions*.csv")):\n                predictions = read_rows(path)\n                if len(predictions) != len(tests):\n                    raise ValueError(f"Test row count differs in {path}")\n                unknown = "unseen" if name == "deepunk" else str(len(known))\n                categories = Counter()\n                first = {}\n                for index, (row, test) in enumerate(zip(predictions, tests)):\n                    gold_unknown = row["gold"] == unknown\n                    if gold_unknown != (test["label"] not in known):\n                        raise ValueError(f"Prediction order or label mapping differs in {path}")\n                    if row["gold"] == row["predicted"]:\n                        category = "correct"\n                    elif gold_unknown:\n                        category = "unknown accepted"\n                    elif row["predicted"] == unknown:\n                        category = "known rejected"\n                    else:\n                        category = "wrong known class"\n                    categories[category] += 1\n                    if category != "correct":\n                        first.setdefault(category, dict(test_row=index, text=test["text"],\n                                                        intent=test["label"], prediction=row["predicted"]))\n                context = dict(mode=mode, trial=trial.parent.name, detector=path.stem)\n                for category in ("correct", "unknown accepted", "known rejected", "wrong known class"):\n                    counts.append(dict(context, category=category, count=categories[category]))\n                for category, example in first.items():\n                    examples.append(dict(context, category=category, **example))\n    write_rows(output / "table_error_counts.csv", counts)\n    if examples:\n        write_rows(output / "error_examples.csv", examples)\n\n\ndef main():\n    """Regenerate a report table using saved experiment evidence."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("table", choices=["compare", "mask", "errors"])\n    parser.add_argument("--upstream", type=Path, default=ROOT / ".bolt")\n    parser.add_argument("--output", type=Path, default=ROOT / "results")\n    args = parser.parse_args()\n    if args.table == "compare":\n        compare(args.output)\n    elif args.table == "mask":\n        mask_effect(args.output)\n    else:\n        errors(args.upstream, args.output)\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'numpy==1.26.4\npandas==2.2.3\nscikit-learn==1.5.2\nmatplotlib==3.9.4\nPyYAML==6.0.2\nnltk==3.9.2\ntensorflow==2.18.1\ntf-keras==2.18.0\ntorch==2.6.0\ntransformers==4.30.2\ndatasets==2.21.0\nfitlog==0.9.15\n', 'published.csv': 'method,known_ratio,K-F1,N-F1\ndeepunk,0.25,37.49,4.16\ndeepunk,0.5,48.99,31.21\nknncon,0.25,63.07,67.07\nknncon,0.5,77.67,58.30\n', 'reproduce.py': '"""Run the released BOLT implementations and audit the BANKING77 label mask."""\n\nimport argparse\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nfrom statistics import mean, stdev\nimport subprocess\nimport sys\nfrom time import monotonic\nfrom urllib.request import urlretrieve\nimport zipfile\n\nROOT = Path(__file__).resolve().parent\nUPSTREAM = "https://github.com/CNIC-DSL/BOLT.git"\nREVISION = "2c935438b5506febcfd23e499e3de81868236b17"\nRATIOS = ("0.25", "0.5")\nMETHODS = {"deepunk": "DeepUnk/experiment.py", "knncon": "KnnCon/run_main.py"}\nMETRICS = ("K-F1", "N-F1", "ACC", "F1")\n\n\ndef read_rows(path, delimiter=","):\n    """Read the author TSV/CSV files without inferring label-mask types."""\n    with path.open(encoding="utf-8", newline="") as handle:\n        return list(csv.DictReader(handle, delimiter=delimiter))\n\n\ndef write_rows(path, rows):\n    """Write a reproducible table with stable column order."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef prepare(source):\n    """Acquire the immutable author release and the DeepUNK embeddings."""\n    if not source.exists():\n        subprocess.run(["git", "clone", UPSTREAM, str(source)], check=True)\n        subprocess.run(["git", "checkout", "--detach", REVISION], cwd=source, check=True)\n    verify_revision(source)\n    glove = source / "pretrained_models" / "glove.6B.300d.txt"\n    glove.parent.mkdir(exist_ok=True)\n    if not glove.exists():\n        archive = glove.with_suffix(".zip")\n        urlretrieve("https://nlp.stanford.edu/data/glove.6B.zip", archive)\n        with zipfile.ZipFile(archive) as embeddings:\n            temporary = glove.with_suffix(".partial")\n            with embeddings.open(glove.name) as original, temporary.open("wb") as target:\n                shutil.copyfileobj(original, target)\n            temporary.replace(glove)\n        archive.unlink()\n\n\ndef verify_revision(source):\n    """Fail if the chosen checkout does not identify the pinned release."""\n    head = (source / ".git" / "HEAD").read_text().strip()\n    if head.startswith("ref: "):\n        ref = head.removeprefix("ref: ")\n        ref_path = source / ".git" / ref\n        if ref_path.exists():\n            head = ref_path.read_text().strip()\n        else:\n            packed = (source / ".git" / "packed-refs").read_text().splitlines()\n            head = next(line.split()[0] for line in packed if line.endswith(" " + ref))\n    if head != REVISION:\n        raise ValueError(f"Expected BOLT {REVISION}, found {head}")\n\n\ndef audit(source):\n    """Verify split alignment and count the effect of the string-to-bool conversion."""\n    data = source / "data" / "banking"\n    rows = []\n    for split in ("train", "dev"):\n        original = read_rows(data / "origin_data" / f"{split}.tsv", "\\t")\n        masked = read_rows(data / "labeled_data/0.1" / f"{split}.tsv", "\\t")\n        if [row["label"] for row in original] != [row["label"] for row in masked]:\n            raise ValueError(f"{split} labels are misaligned")\n        if {row["labeled"] for row in masked} != {"True", "False"}:\n            raise ValueError(f"Unexpected mask values in {split}")\n        for ratio in RATIOS:\n            for fold in range(5):\n                path = data / f"label/fold5/part{fold}/label_known_{ratio}.list"\n                known = set(path.read_text(encoding="utf-8").splitlines())\n                candidates = [row for row in masked if row["label"] in known]\n                selected = [row for row in candidates if row["labeled"] == "True"]\n                if {row["label"] for row in selected} != known:\n                    raise ValueError(f"Missing labeled examples for {ratio}, fold {fold}, {split}")\n                rows.append(dict(split=split, known_ratio=ratio, fold=fold,\n                                 classes=len(known), released_knncon=len(candidates),\n                                 corrected_knncon=len(selected)))\n    return rows\n\n\ndef replace_once(source, needle, replacement):\n    """Anchor instrumentation to the inspected source revision."""\n    if source.count(needle) != 1:\n        raise ValueError(f"Unexpected upstream source around {needle}")\n    return source.replace(needle, replacement)\n\n\ndef stage_method(source, name, corrected, work):\n    """Copy a method to scratch and add prediction exports and the optional mask fix."""\n    entry = Path(METHODS[name])\n    location = work / entry.parent\n    shutil.copytree(source / "code/openset/baselines" / entry.parent, location,\n                    dirs_exist_ok=True)\n    if name == "deepunk":\n        path = location / entry.name\n        code = path.read_text(encoding="utf-8")\n        code = replace_once(code, "    report = classification_report(",\n            \'    os.makedirs(args.save_results_path, exist_ok=True)\\n\'\n            \'    pd.DataFrame({"gold": y_test_mask, "predicted": y_pred, \'\n            \'"raw_gold": df_test["label"]}).to_csv(\\n\'\n            \'        os.path.join(args.save_results_path, "predictions.csv"), index=False\\n\'\n            \'    )\\n\\n    report = classification_report(\')\n    else:\n        path = location / "evaluate.py"\n        code = path.read_text(encoding="utf-8")\n        code = replace_once(code, "                df_to_save = pd.DataFrame([final_results])",\n            \'                pd.DataFrame({"gold": labels_test, "predicted": predict_label_test}).to_csv(\\n\'\n            \'                    os.path.join(self.data_args.save_results_path, \'\n            \'f"predictions_{setting}.csv"), index=False\\n\'\n            \'                )\\n\'\n            \'                df_to_save = pd.DataFrame([final_results])\')\n        code = replace_once(code, \'"seed","ACC","F1","K-F1","N-F1","args"]\',\n                            \'"seed","ACC","F1","K-F1","N-F1","args","ood_method"]\')\n        if corrected:\n            runner = location / entry.name\n            runner_code = runner.read_text(encoding="utf-8")\n            for frame in ("df_train", "df_valid"):\n                runner_code = replace_once(runner_code, f\'{frame}["labeled"].astype(bool)\',\n                                           f\'{frame}["labeled"].eq("True")\')\n            runner.write_text(runner_code, encoding="utf-8")\n    path.write_text(code, encoding="utf-8")\n    return location / entry.name\n\n\ndef run_trial(source, output, name, ratio, fold, mode):\n    """Run one isolated fold and record command, elapsed time, and console output."""\n    corrected = mode == "mask"\n    dest = output / mode / f"{name}_{ratio}_{fold}"\n    dest.mkdir(parents=True, exist_ok=True)\n    work = output / "work" / mode / dest.name\n    args = dict(config=source / f"configs/openset/{name}.yaml", dataset="banking",\n                known_cls_ratio=ratio, labeled_ratio=0.1, fold_idx=fold, fold_num=5,\n                fold_type="fold", seed=2025, num_train_epochs=1 if mode == "quick" else 50,\n                output_dir=work / "checkpoints", save_results_path=dest)\n    if name == "knncon":\n        args["bert_model"] = "google-bert/bert-base-uncased"\n    entry = stage_method(source, name, corrected, work)\n    interpreter = os.environ.get("BOLT_TF_PYTHON", sys.executable) if name == "deepunk" else sys.executable\n    command = [interpreter, str(entry)]\n    for key, value in args.items():\n        command.extend((f"--{key}", str(value)))\n    env = dict(os.environ, PYTHONHASHSEED="2025", TF_USE_LEGACY_KERAS="1",\n               PYTHONUNBUFFERED="1", WANDB_DISABLED="true")\n    if name == "deepunk" and env.get("BOLT_TF_LIBS"):\n        env["LD_LIBRARY_PATH"] = env["BOLT_TF_LIBS"]\n    if (dest / "run.json").exists():\n        raise FileExistsError(f"Run already completed in {dest}. Choose another output directory.")\n    for path in dest.glob("*.csv"):\n        path.unlink()\n    start = monotonic()\n    print(f"Running {dest.name} ({mode})", flush=True)\n    with (dest / "run.log").open("w", encoding="utf-8") as log:\n        subprocess.run(command, cwd=source, env=env, stdout=log, stderr=subprocess.STDOUT,\n                       check=True)\n    rows = read_rows(dest / "results.csv")\n    expected = 1 if name == "deepunk" else 2\n    if len(rows) != expected:\n        raise ValueError(f"Expected {expected} result rows in {dest}")\n    run_info = dict(command=command, elapsed_seconds=round(monotonic() - start, 2),\n                    upstream=REVISION, mode=mode)\n    (dest / "run.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")\n\n\ndef run_table(output, mode, folds):\n    """Aggregate only complete groups and retain both KNNCon detectors."""\n    rows = []\n    for name in METHODS:\n        if mode == "mask" and name == "deepunk":\n            continue\n        for ratio in RATIOS:\n            by_detector = {}\n            for fold in folds:\n                directory = output / mode / f"{name}_{ratio}_{fold}"\n                if not (directory / "run.json").is_file():\n                    raise ValueError(f"Incomplete trial {directory}")\n                for row in read_rows(directory / "results.csv"):\n                    by_detector.setdefault(row.get("ood_method", "lof"), []).append(row)\n            for detector, trials in by_detector.items():\n                if len(trials) != len(folds):\n                    raise ValueError(f"Missing detector output for {name}, {ratio}, {detector}")\n                result = dict(method=name, known_ratio=ratio, detector=detector,\n                              folds=len(folds))\n                for metric in METRICS:\n                    values = [float(trial[metric]) for trial in trials]\n                    if name == "deepunk":\n                        values = [100 * value for value in values]\n                    result[metric] = round(mean(values), 3)\n                    result[f"{metric}_fold_sd"] = round(stdev(values), 3) if len(values) > 1 else ""\n                rows.append(result)\n    write_rows(output / f"table_{mode}.csv", rows)\n\n\ndef main():\n    """Dispatch preparation, audit, reduced run, full grid, or mask sensitivity."""\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=["prepare", "audit", "quick", "full", "mask", "check"])\n    parser.add_argument("--upstream", type=Path, default=ROOT / ".bolt")\n    parser.add_argument("--output", type=Path, default=ROOT / "results")\n    options = parser.parse_args()\n    source, output = options.upstream.resolve(), options.output.resolve()\n    if options.action == "prepare":\n        prepare(source)\n        return\n    verify_revision(source)\n    if options.action in {"audit", "check"}:\n        rows = audit(source)\n        if options.action == "audit":\n            write_rows(output / "table_mask_audit.csv", rows)\n        else:\n            import tempfile\n            with tempfile.TemporaryDirectory() as scratch:\n                for method in METHODS:\n                    entry = stage_method(source, method, True, Path(scratch))\n                    compile(entry.read_text(encoding="utf-8"), str(entry), "exec")\n                    if method == "knncon":\n                        evaluator = entry.with_name("evaluate.py")\n                        compile(evaluator.read_text(encoding="utf-8"), str(evaluator), "exec")\n            print(f"Passed split alignment, mask coverage, and patch checks ({len(rows)} split cases)")\n        return\n    if options.action == "quick":\n        run_trial(source, output, "knncon", "0.25", 0, "quick")\n        return\n    folds = range(5)\n    for name in METHODS:\n        if options.action == "mask" and name == "deepunk":\n            continue\n        for ratio in RATIOS:\n            for fold in folds:\n                dest = output / options.action / f"{name}_{ratio}_{fold}"\n                if not (dest / "run.json").exists():\n                    run_trial(source, output, name, ratio, fold, options.action)\n    run_table(output, options.action, folds)\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, source in sources.items():
    (root / name).write_text(source, encoding="utf-8")

## Set up separate GPU environments
KNNCon uses PyTorch. DeepUNK uses TensorFlow with its own pinned CUDA libraries. The checks below execute actual GPU operations in both environments before any training starts.

In [8]:
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.9.22"], check=True)

requirements = (root / "requirements.txt").read_text().splitlines()
common = ["numpy", "pandas", "scikit-learn", "matplotlib", "PyYAML"]
torch_packages = common + ["torch", "transformers", "datasets", "fitlog"]
tf_packages = common + ["nltk", "tf-keras"]

def install(name, packages):
    path = root / name
    python = path / "bin/python"
    if not python.exists():
        subprocess.run(["uv", "venv", "--python", "3.11", str(path)], check=True)
    subprocess.run(
        ["uv", "pip", "install", "--python", str(python), *packages], check=True
    )
    return python

python = install(".venv-torch", [
    line for line in requirements if line.split("==")[0] in torch_packages
])
tf_python = install(".venv-tf", [
    line for line in requirements if line.split("==")[0] in tf_packages
] + ["tensorflow[and-cuda]==2.18.1"])

base_env = dict(os.environ, MPLBACKEND="Agg", TF_USE_LEGACY_KERAS="1",
                CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
if base_env.get("NVIDIA_VISIBLE_DEVICES") in {"none", "void"}:
    base_env.pop("NVIDIA_VISIBLE_DEVICES")
library_dirs = sorted((tf_python.parent.parent / "lib/python3.11/site-packages/nvidia").glob("*/lib"))
assert library_dirs, "TensorFlow CUDA libraries were not installed."
tf_libs = os.pathsep.join([*(str(path) for path in library_dirs),
                           base_env.get("LD_LIBRARY_PATH", "")]).rstrip(os.pathsep)
tf_env = dict(base_env, LD_LIBRARY_PATH=tf_libs)
run_env = dict(base_env, BOLT_TF_PYTHON=str(tf_python), BOLT_TF_LIBS=tf_libs)

torch_check = """
import torch
assert torch.cuda.is_available(), "PyTorch cannot see the GPU"
a = torch.ones((32, 32), device="cuda")
print("PyTorch GPU:", torch.cuda.get_device_name(), (a @ a).device, flush=True)
"""
tf_check = """
import tensorflow as tf
assert tf.config.list_physical_devices("GPU"), "TensorFlow cannot see the GPU"
tf.config.set_soft_device_placement(False)
with tf.device("/GPU:0"):
    layer = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16, dropout=0.5))
    inputs = tf.ones((4, 16, 8))
    with tf.GradientTape() as tape:
        output = layer(inputs, mask=tf.sequence_mask([16, 12, 8, 4], 16), training=True)
        loss = tf.reduce_sum(output)
    gradients = tape.gradient(loss, layer.trainable_variables)
    assert all(gradient is not None for gradient in gradients)
    assert "GPU:0" in output.device, output.device
    print("TensorFlow LSTM GPU:", output.device, flush=True)
"""
with (results / "environment.txt").open("w") as info:
    for command in (["nvidia-smi"], [str(python), "-VV"],
                    ["uv", "pip", "freeze", "--python", str(python)],
                    ["uv", "pip", "freeze", "--python", str(tf_python)]):
        subprocess.run(command, stdout=info, stderr=subprocess.STDOUT, check=True)

for interpreter, check, env in [(python, torch_check, base_env), (tf_python, tf_check, tf_env)]:
    checked = subprocess.run([str(interpreter), "-c", check], env=env,
                             text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(checked.stdout, flush=True)
    with (results / "environment.txt").open("a") as info:
        info.write(checked.stdout)
    checked.check_returncode()

PyTorch GPU: Tesla T4 cuda:0

2026-11-18 06:14:17.502819: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794982457.521648    5069 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794982457.527550    5069 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-18 06:14:17.547484: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-11-18 06:14:27.574935: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:4

In [9]:
import shutil
from urllib.request import urlretrieve
from zipfile import ZipFile

upstream = root / ".bolt-pinned"
revision = "2c935438b5506febcfd23e499e3de81868236b17"

if not (upstream / ".git").is_dir():
    upstream.mkdir(exist_ok=True)
    subprocess.run(["git", "init", str(upstream)], check=True)

head = subprocess.run(
    ["git", "rev-parse", "--verify", "HEAD"],
    cwd=upstream, capture_output=True, text=True,
)
if head.stdout.strip() != revision:
    subprocess.run(
        ["git", "fetch", "--depth", "1",
         "https://github.com/CNIC-DSL/BOLT.git", revision],
        cwd=upstream, check=True,
    )
    subprocess.run(
        ["git", "checkout", "--detach", "FETCH_HEAD"],
        cwd=upstream, check=True,
    )

def run(script, action):
    if script == "reproduce.py" and action == "full":
        glove = upstream / "pretrained_models" / "glove.6B.300d.txt"
        if not glove.is_file():
            glove.parent.mkdir(exist_ok=True)
            archive = glove.parent / "glove.6B.zip"
            partial = glove.with_suffix(".partial")
            print("Downloading GloVe for DeepUNK...", flush=True)
            urlretrieve(
                "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip",
                archive,
            )
            with ZipFile(archive) as bundle:
                with bundle.open(glove.name) as source, partial.open("wb") as target:
                    shutil.copyfileobj(source, target)
            partial.replace(glove)
            archive.unlink()

    subprocess.run(
        [str(python), str(root / script), action,
         "--upstream", str(upstream), "--output", str(results)],
        cwd=root, env=run_env,
        check=True,
    )

run("reproduce.py", "check")
run("reproduce.py", "audit")
print("Setup complete. Run the Reduced run cell next.")

Setup complete. Run the Reduced run cell next.


In [10]:
subprocess.run(
    ["uv", "pip", "install", "--python", str(python), "pyarrow==17.0.0"],
    check=True,
)

subprocess.run(
    [str(python), "-c",
     'import numpy, pyarrow; from datasets import Dataset; '
     'assert len(Dataset.from_dict({"text": ["check"]})) == 1; '
     'print("Dataset check passed:", numpy.__version__, pyarrow.__version__)'],
    env=run_env,
    check=True,
)

with (results / "environment.txt").open("a") as info:
    info.write("\nPyTorch environment after PyArrow fix:\n")
    subprocess.run(
        ["uv", "pip", "freeze", "--python", str(python)],
        stdout=info,
        check=True,
    )

In [3]:
from pathlib import Path
from google.colab import files

root = Path("/content/bolt-reproduction-gpu")
results = root / "results"
results.mkdir(parents=True, exist_ok=True)

In [5]:
from io import BytesIO
from zipfile import ZipFile
from google.colab import files


uploaded = files.upload()
with ZipFile(BytesIO(uploaded["BOLT-GPU-results.zip"])) as saved:
    for item in saved.infolist():
        parts = item.filename.split("/")
        if (
            item.is_dir()
            or parts[0] not in {"full", "quick", "mask"}
            or ".." in parts
        ):
            continue
        target = results.joinpath(*parts)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(saved.read(item))

print("Restored completed full trials:", len(list((results / "full").glob("*/run.json"))))

Saving BOLT-GPU-results.zip to BOLT-GPU-results.zip
Restored completed full trials: 10


## Reduced run

In [6]:
quick = results / "quick/knncon_0.25_0"
try:
    if not (quick / "run.json").exists():
        run("reproduce.py", "quick")
except subprocess.CalledProcessError:
    if (quick / "run.log").exists():
        print("\n".join((quick / "run.log").read_text(errors="replace").splitlines()[-35:]))
    raise
print((quick / "run.json").read_text())
print((quick / "results.csv").read_text())

{
  "command": [
    "/content/bolt-reproduction-gpu/.venv-torch/bin/python",
    "/content/bolt-reproduction-gpu/results/work/quick/knncon_0.25_0/KnnCon/run_main.py",
    "--config",
    "/content/bolt-reproduction-gpu/.bolt-pinned/configs/openset/knncon.yaml",
    "--dataset",
    "banking",
    "--known_cls_ratio",
    "0.25",
    "--labeled_ratio",
    "0.1",
    "--fold_idx",
    "0",
    "--fold_num",
    "5",
    "--fold_type",
    "fold",
    "--seed",
    "2025",
    "--num_train_epochs",
    "1",
    "--output_dir",
    "/content/bolt-reproduction-gpu/results/work/quick/knncon_0.25_0/checkpoints",
    "--save_results_path",
    "/content/bolt-reproduction-gpu/results/quick/knncon_0.25_0",
    "--bert_model",
    "google-bert/bert-base-uncased"
  ],
  "elapsed_seconds": 176.59,
  "upstream": "2c935438b5506febcfd23e499e3de81868236b17",
  "mode": "quick"
}
method,dataset,known_cls_ratio,labeled_ratio,cluster_num_factor,seed,ACC,F1,K-F1,N-F1,args,ood_method
KnnCon,banking,0.25,0.

## Full reproduction and corrected-mask analysis

In [7]:
run("reproduce.py", "full")
run("analyze.py", "compare")
print((results / "table_comparison.csv").read_text())

KeyboardInterrupt: 

In [12]:
from pathlib import Path
import json
import os

root = Path("/content/bolt-reproduction-gpu")
results = root / "results"
runner_path = root / "reproduce.py"

# Load an isolated five-epoch runner without changing the full experiment.
source = runner_path.read_text(encoding="utf-8")
old = 'num_train_epochs=1 if mode == "quick" else 50'
assert source.count(old) == 1, "Runner differs from the expected version."
short_source = source.replace(old, 'num_train_epochs=1 if mode == "quick" else 5')
short_runner = root / "reproduce_short.py"
short_runner.write_text(short_source, encoding="utf-8")
namespace = {"__name__": "short_experiment", "__file__": str(short_runner)}
exec(compile(short_source, str(short_runner), "exec"), namespace)

from types import SimpleNamespace
namespace["sys"] = SimpleNamespace(executable=str(python))


# Requires the setup cells and both successful GPU checks in this session.
assert python.is_file(), "Run the setup cells first."
assert "BOLT_TF_PYTHON" in run_env, "Run the GPU setup cell first."
namespace["verify_revision"](upstream)
output = results / "short_5epoch"
for mode in ("full", "mask"):
    trial = output / mode / "knncon_0.25_0"
    if not (trial / "run.json").exists():
        print(f"Starting five-epoch KNNCon: {mode}", flush=True)
        # run_trial starts a subprocess. Propagate the tested GPU environment.
        previous = {key: os.environ.get(key) for key in run_env}
        os.environ.update(run_env)
        try:
            namespace["run_trial"](upstream, output, "knncon", "0.25", 0, mode)
        except Exception:
            log = trial / "run.log"
            if log.exists():
                print("\n".join(log.read_text(errors="replace").splitlines()[-35:]))
            raise
        finally:
            for key, value in previous.items():
                if value is None:
                    os.environ.pop(key, None)
                else:
                    os.environ[key] = value
    print((trial / "results.csv").read_text())
    print((trial / "run.json").read_text())

print("Paired five-epoch experiment complete. Run Download saved evidence.")


Starting five-epoch KNNCon: full
Running knncon_0.25_0 (full)
method,dataset,known_cls_ratio,labeled_ratio,cluster_num_factor,seed,ACC,F1,K-F1,N-F1,args,ood_method
KnnCon,banking,0.25,0.1,1.0,2025,75.75,70.39,69.8,81.59,"{""output_dir"": ""/content/bolt-reproduction-gpu/results/short_5epoch/work/full/knncon_0.25_0/checkpoints"", ""overwrite_output_dir"": true, ""do_train"": true, ""do_eval"": true, ""do_predict"": true, ""model_parallel"": false, ""evaluation_strategy"": ""epoch"", ""prediction_loss_only"": false, ""per_device_train_batch_size"": 16, ""per_device_eval_batch_size"": 16, ""per_gpu_train_batch_size"": null, ""per_gpu_eval_batch_size"": null, ""gradient_accumulation_steps"": 1, ""eval_accumulation_steps"": null, ""learning_rate"": 5e-05, ""weight_decay"": 0.0001, ""adam_beta1"": 0.9, ""adam_beta2"": 0.98, ""adam_epsilon"": 1e-08, ""max_grad_norm"": 1.0, ""num_train_epochs"": 5, ""max_steps"": -1, ""warmup_steps"": 72.0, ""logging_dir"": ""runs/Nov18_06-17-41_aeb5159ce6a8""

In [13]:
run("reproduce.py", "mask")
run("analyze.py", "mask")
run("analyze.py", "errors")
print((results / "table_mask_effect.csv").read_text())

KeyboardInterrupt: 

## Download saved evidence
This cell can also export completed trials before the entire grid finishes.

In [14]:
with ZipFile("/content/BOLT-GPU-results.zip", "w") as bundle:
    for path in results.rglob("*"):
        relative = path.relative_to(results)
        if path.is_file() and relative.parts[0] != "work":
            bundle.write(path, relative)
    for name in sources:
        bundle.write(root / name, "source/" + name)
files.download("/content/BOLT-GPU-results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>